**© Copyright AIDENTIFY. All rights reserved.**

# Part 3 | Session 25: vLLM 서빙 — PagedAttention과 OpenAI 호환 API

---

### 📋 학습 목표

- 🔹 vLLM의 핵심 기술(PagedAttention, Continuous Batching)을 이해합니다
- 🔹 vLLM Python API로 오프라인 추론을 실행합니다
- 🔹 vLLM OpenAI 호환 API 서버를 로컬에서 구축합니다
- 🔹 동일 모델을 Transformers 직접 추론과 vLLM 서빙으로 처리량 비교합니다
- 🔹 GPU VRAM에 맞는 vLLM 설정(모델 / dtype / max-model-len)을 선택합니다

### 📦 필요 라이브러리

```
vllm, openai, torch, transformers, requests
```

### ⏱️ 예상 소요 시간: 약 90분

### 🖥️ 실습 환경

| 환경 | GPU | 실습 모델 | 비고 |
|------|-----|-----------|------|
| RTX 4060 (8GB) | 1.5B FP16 | Qwen2.5-1.5B-Instruct | 본 과정 기본 |
| RTX 4090 / TITAN RTX (24GB+) | 7B FP16 | Qwen2.5-7B-Instruct | 멀티GPU 가능 |

---

> 💡 Session 21~24의 양자화 모델(AWQ/GPTQ/QLoRA)을 **production에서 어떻게 서빙하는가**가 본 세션의 주제입니다.

---

## 1️⃣ vLLM 소개

**vLLM**은 대규모 언어 모델을 빠르게 서빙하기 위한 오픈소스 추론 엔진입니다 (UC Berkeley 2023).

### 🚀 핵심 기술

- 🔹 **PagedAttention** — GPU 메모리를 페이지 단위로 관리해 KV cache 메모리 낭비를 **90% 이상** 절감
- 🔹 **Continuous Batching** — 요청이 끝나는 즉시 빈 슬롯에 다음 요청 투입 → 대기시간 최소화
- 🔹 **OpenAI 호환 API** — `openai` 라이브러리로 바로 호출 (기존 코드 그대로 재사용)
- 🔹 **양자화 통합** — AWQ / GPTQ / FP8 / Marlin 모델 직접 서빙

### 📊 Static vs Continuous Batching

```
Static Batching (기존):
  요청1: [████████████████████]  → 모든 요청이 끝날 때까지 대기
  요청2: [████████____대기____]  → 짧은 응답도 긴 응답 끝까지 대기
  요청3: [██████████████______]  → GPU 활용률 낮음

Continuous Batching (vLLM):
  요청1: [████████████████████]  → 끝나면 즉시 새 요청 투입
  요청2: [████████] → 요청4: [████████████]
  요청3: [██████████████] → 요청5: [██████]   → GPU 항상 바쁨
```

---

In [ ]:
# 📊 vLLM vs Ollama vs Transformers 서빙 비교
print("📊 LLM 서빙 프레임워크 비교")
print("=" * 78)

comparisons = [
    ("설치 난이도",   "보통",            "매우 쉬움",       "매우 쉬움"),
    ("처리량 (QPS)",  "높음 (10x+)",     "보통 (3x)",       "기준 (1x)"),
    ("동시 요청",     "수백 개",          "수십 개",          "1개"),
    ("배칭",          "Continuous",      "Static",          "없음"),
    ("모델 포맷",     "HF (FP16/AWQ)",   "GGUF (양자화)",   "HF (모든 형식)"),
    ("API 호환",      "OpenAI",          "OpenAI",          "없음"),
    ("최소 VRAM",     "6GB+ (1.5B)",     "4GB+ (1.5B)",     "2GB+"),
    ("주요 용도",     "프로덕션 서빙",    "개발/테스트",      "실험/학습"),
]

print(f"{'항목':<15} {'vLLM':>17} {'Ollama':>17} {'Transformers':>17}")
print("-" * 78)
for row in comparisons:
    print(f"{row[0]:<15} {row[1]:>17} {row[2]:>17} {row[3]:>17}")

print(f"\n💡 선택 가이드:")
print(f"   🔹 프로덕션 서빙 (다수 동시 사용자) → vLLM")
print(f"   🔹 로컬 개발/테스트            → Ollama")
print(f"   🔹 연구/커스텀 추론              → Transformers")

---

## 2️⃣ vLLM 설치 & 환경 확인

vLLM은 pip으로 설치합니다. **CUDA 12.1 이상**, **Python 3.9+** 가 필요합니다.

```bash
# 터미널에서
pip install vllm
```

> ⚠️ 설치 시간이 길 수 있습니다 (5~10분).
> 설치 실패 시 CUDA / Python 버전을 확인하세요.

In [ ]:
# 🔧 vLLM 설치 확인 + GPU 환경 점검
import sys, torch

print("🔧 환경 점검")
print("=" * 50)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

vllm_available = False
default_model = None
vram_gb = 0

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f}GB")

    if vram_gb >= 20:
        default_model = "Qwen/Qwen2.5-7B-Instruct"
        print(f"\n✅ VRAM {vram_gb:.0f}GB → 7B FP16 서빙 가능")
    elif vram_gb >= 6:
        default_model = "Qwen/Qwen2.5-1.5B-Instruct"
        print(f"\n✅ VRAM {vram_gb:.0f}GB → 1.5B FP16 서빙 가능")
    else:
        print(f"\n⚠️ VRAM {vram_gb:.0f}GB → vLLM 사용 어려움")

    if default_model:
        print(f"📌 기본 실습 모델: {default_model}")
else:
    print("⚠️ GPU를 사용할 수 없습니다.")

print()
try:
    import vllm
    print(f"✅ vLLM: {vllm.__version__}")
    vllm_available = True
except ImportError:
    print("❌ vLLM이 설치되어 있지 않습니다.")
    print("   터미널: pip install vllm")

---

## 3️⃣ vLLM 사용 방식: 오프라인 추론 vs API 서버

vLLM은 **두 가지 방식**으로 사용할 수 있습니다.

### 방식 A: 오프라인 추론 (Python API)

코드 안에서 직접 모델을 로드하고 추론. 서버 불필요.

```python
from vllm import LLM, SamplingParams
llm = LLM(model="Qwen/Qwen2.5-7B-Instruct", dtype="float16")
outputs = llm.generate(["한국의 수도는?", "파이썬이란?"], SamplingParams(max_tokens=128))
```

- 코드 간단, 스크립트/노트북 실험에 적합
- 배치 처리 효율적
- **단일 프로세스만** (외부 호출 불가)

### 방식 B: API 서버 (OpenAI 호환)

터미널에서 서버 띄우고 HTTP API로 호출.

```bash
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-7B-Instruct --dtype float16 --port 9100
```

```python
from openai import OpenAI
client = OpenAI(base_url="http://localhost:9100/v1", api_key="not-needed")
response = client.chat.completions.create(model="...", messages=[...])
```

- OpenAI API와 **동일 인터페이스** → 기존 코드 재사용
- Continuous Batching으로 동시 요청 자동 처리
- 여러 클라이언트 동시 접속 가능 (production 적합)

```
⚠️ 같은 GPU에서 A·B를 동시에 실행 불가!
   → 방식 A 실습 후 커널 재시작 → 방식 B 실습
```

> 본 노트북은 **A → B 순서**로 실습합니다.

---

### ▶️ 방식 A 실습: 오프라인 추론 (Python API)

아래 **3개 코드 셀**이 방식 A 실습입니다. 서버 없이 노트북 안에서 직접 모델을 로드해 추론합니다.

| 셀 | 내용 |
|----|------|
| ① 모델 로딩 + 단일 프롬프트 | `LLM()` 로 모델 로드 후 1개 프롬프트 추론 |
| ② 배칭 성능 비교 | 개별(1×5) vs 배치(5동시) 처리량 비교 |
| ③ GPU 메모리 정리 | 방식 B로 넘어가기 전 VRAM 해제 |

> ⚠️ 방식 A를 마치면 **③ 셀 실행 → 커널 재시작** 후 아래 `4️⃣ 방식 B`로 진행하세요.

In [ ]:
# 🚀 vLLM 오프라인 추론 — 모델 로딩 + 단일 프롬프트
import os, time

# ⚠️ CUDA toolkit(nvcc)이 없는 환경(예: AWS g5.xlarge)에서는 FlashInfer 샘플러가
#    런타임 JIT 컴파일에 실패합니다("Could not find nvcc").
#    네이티브 PyTorch 샘플러를 쓰도록 강제하면 nvcc 없이도 정상 동작합니다.
#    (반드시 vLLM 임포트/엔진 생성 전에 설정해야 적용됩니다.)
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

if not vllm_available:
    print("⚠️ vLLM 미설치 — 셀 건너뜀")
elif not torch.cuda.is_available():
    print("⚠️ GPU 미사용 — 셀 건너뜀")
else:
    from vllm import LLM, SamplingParams

    if vram_gb >= 20:
        model_name = "Qwen/Qwen2.5-7B-Instruct"
        max_len, mem_util = 2048, 0.85
    else:
        model_name = "Qwen/Qwen2.5-1.5B-Instruct"
        max_len, mem_util = 1024, 0.80

    print(f"🚀 모델 로딩: {model_name}")
    print(f"   max_model_len={max_len}, gpu_memory_utilization={mem_util}")

    start = time.time()
    llm = LLM(
        model=model_name,
        dtype="float16",
        max_model_len=max_len,
        gpu_memory_utilization=mem_util,
        enforce_eager=True,  # Turing(TITAN RTX 등)에서 FA2 미지원 시 필요
    )
    print(f"✅ 로딩 완료 ({time.time()-start:.1f}초)")

    sampling = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=128)
    prompts = ["인공지능의 미래에 대해 간단히 설명해주세요."]

    print("\n📝 단일 프롬프트 추론")
    s = time.time()
    outputs = llm.generate(prompts, sampling)
    elapsed = time.time() - s

    for o in outputs:
        gen = o.outputs[0].text
        n = len(o.outputs[0].token_ids)
        print(f"\n❓ {o.prompt}")
        print(f"💬 {gen}")
        print(f"\n📊 {n} tokens / {elapsed:.2f}s = {n/elapsed:.1f} tok/s")

In [ ]:
# 📊 배칭 성능 비교: 개별 추론(5회) vs 배치 추론(5개 동시)
if not vllm_available or not torch.cuda.is_available():
    print("⚠️ 건너뜀")
else:
    try:
        llm
    except NameError:
        print("⚠️ 이전 셀에서 llm을 먼저 로드하세요.")
    else:
        sampling = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=128)
        batch_prompts = [
            "파이썬의 장점을 3가지 알려주세요.",
            "머신러닝과 딥러닝의 차이점은?",
            "한국의 대표적인 전통 음식 3가지는?",
            "클라우드 컴퓨팅이란 무엇인가요?",
            "좋은 코드를 작성하는 방법은?",
        ]

        print("📊 배칭 성능 비교")
        print("=" * 60)

        # 1) 개별 (1개씩 × 5)
        s = time.time()
        ind_tok = 0
        for p in batch_prompts:
            o = llm.generate([p], sampling)
            ind_tok += len(o[0].outputs[0].token_ids)
        ind_t = time.time() - s

        # 2) 배치 (5개 한 번에)
        s = time.time()
        bouts = llm.generate(batch_prompts, sampling)
        bat_t = time.time() - s
        bat_tok = sum(len(o.outputs[0].token_ids) for o in bouts)

        print(f"\n🔸 개별 추론 (1×5): {ind_t:.2f}s, {ind_tok} tok, {ind_tok/ind_t:.1f} tok/s")
        print(f"🔹 배치 추론 (5동시): {bat_t:.2f}s, {bat_tok} tok, {bat_tok/bat_t:.1f} tok/s")
        print(f"\n🚀 배치 처리 속도 향상: {ind_t/bat_t:.1f}x")
        print("\n💡 Continuous Batching이 GPU 활용률을 높여 전체 처리량을 크게 증가시킵니다.")

        # 💾 방식 B와 비교하기 위해 방식 A 결과를 파일로 저장
        #    (커널을 재시작해도 파일은 남으므로, 방식 B 셀에서 불러와 비교합니다.)
        import json
        method_a = {
            "method": "A_offline",
            "model": model_name,
            "n_prompts": len(batch_prompts),
            "individual_throughput_tok_s": round(ind_tok / ind_t, 1),
            "batch_throughput_tok_s": round(bat_tok / bat_t, 1),
            "batch_speedup_x": round(ind_t / bat_t, 2),
        }
        with open("method_a_result.json", "w") as f:
            json.dump(method_a, f, ensure_ascii=False, indent=2)
        print("\n💾 방식 A 결과 저장: method_a_result.json (→ 방식 B에서 비교에 사용)")

In [ ]:
# 🧹 GPU 메모리 정리 — 방식 B(서버)로 넘어가기 전 반드시 실행
# 방식 A에서 로드한 모델이 VRAM을 점유한 채로 남아 있으면, 같은 GPU에 방식 B 서버를
# 띄울 때 "Free memory on device cuda:0 ... less than desired" 오류로 서버가 안 뜹니다.
import gc

try:
    llm
    _have_llm = True
except NameError:
    _have_llm = False

if _have_llm:
    try:
        # vLLM v1은 엔진을 별도 프로세스에서 돌리므로 명시적으로 종료해야 VRAM이 빠집니다.
        llm.llm_engine.shutdown()
    except Exception:
        pass
    del llm
    print("✅ vLLM 모델/엔진 해제 요청")
else:
    print("ℹ️ 해제할 llm 객체가 없습니다.")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()  # 프로세스 합산이 아닌 '장치 전체' 가용 VRAM
    print(f"✅ GPU 사용 가능 VRAM: {free_b/1024**3:.1f} / {total_b/1024**3:.1f} GiB")

print("\n⚠️ vLLM 엔진은 별도 프로세스라 노트북 안에서 완전히 해제되지 않을 수 있습니다.")
print("   위 '사용 가능 VRAM'이 모델 크기(7B≈15GB)만큼 회복되지 않았다면,")
print("   커널 재시작(Kernel → Restart) 후 방식 B로 진행하세요. (가장 확실)")

---

## 4️⃣ 방식 B 실습: vLLM OpenAI 호환 API 서버

서버를 띄우면 `openai` 라이브러리로 바로 호출 가능합니다.
(앞의 `방식 A`는 노트북 내부 추론, 여기 `방식 B`는 별도 서버 + HTTP 호출입니다.)

### 🔄 서버 방식의 장점

- 모델을 **한 번만 로드** → 여러 클라이언트가 공유
- 동시 요청 자동 배칭 → 높은 처리량
- OpenAI 호환 → 기존 코드 그대로 사용

### ⚠️ 아래 셀 실행 전 순서

```
┌──────────────────────────────────────────────────────────────┐
│ Step 1. 노트북 커널 재시작 (Kernel → Restart Kernel)            │
│         → 오프라인 추론에서 사용한 GPU 메모리 해제              │
│                                                              │
│ Step 2. 터미널에서 vLLM API 서버 실행 (다음 셀 참고)            │
│                                                              │
│ Step 3. "Uvicorn running on ..." 메시지 확인 후                │
│         아래 코드 셀 실행                                       │
└──────────────────────────────────────────────────────────────┘
```

> 💡 **CUDA toolkit(nvcc)이 없는 환경(예: AWS g5.xlarge)에서는** 서버 실행 명령
> 앞에 `VLLM_USE_FLASHINFER_SAMPLER=0` 을 붙이세요. 안 붙이면 FlashInfer 샘플러가
> 런타임 JIT 컴파일에 실패해 `Could not find nvcc` 에러로 서버가 뜨지 않습니다.
> (아래 명령어에 이미 포함되어 있습니다. nvcc가 설치된 환경에서는 빼도 됩니다.)

### 📋 Step 2 서버 실행 명령어

**VRAM 20GB+ (RTX 4090 / A10G / A100 등):**
```bash
VLLM_USE_FLASHINFER_SAMPLER=0 \
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-7B-Instruct \
    --dtype float16 \
    --max-model-len 2048 \
    --gpu-memory-utilization 0.85 \
    --enforce-eager \
    --port 9100
```

**RTX 4060 (8GB):**
```bash
VLLM_USE_FLASHINFER_SAMPLER=0 \
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --dtype float16 \
    --max-model-len 1024 \
    --gpu-memory-utilization 0.80 \
    --enforce-eager \
    --port 9100
```

> 서버 시작까지 **수 분** 걸립니다. `Uvicorn running on http://0.0.0.0:9100` 메시지를 기다리세요.

---

In [ ]:
# ⚙️ vLLM API 서버 주요 파라미터
params = [
    ("--model",                   "HuggingFace 모델 이름",           "필수"),
    ("--dtype",                   "데이터 타입 (float16/bfloat16/auto)", "auto"),
    ("--max-model-len",           "최대 시퀀스 길이",                  "모델 기본값"),
    ("--gpu-memory-utilization",  "GPU 메모리 사용 비율 (0~1)",       "0.9"),
    ("--tensor-parallel-size",    "Tensor 병렬 GPU 수 (멀티GPU)",     "1"),
    ("--max-num-seqs",            "동시 처리 최대 시퀀스 수",          "256"),
    ("--quantization",            "양자화 (awq / gptq / squeezellm)", "없음"),
    ("--port",                    "API 서버 포트",                    "8000"),
    ("--host",                    "바인딩 호스트",                    "0.0.0.0"),
    ("--served-model-name",       "API에서 사용할 모델 별칭",          "--model과 동일"),
    ("--enforce-eager",           "FlashAttention 비활성화 (구형 GPU)", "False"),
]

print("⚙️ vLLM API 서버 주요 파라미터")
print("=" * 78)
print(f"{'파라미터':<28} {'설명':<32} {'기본값':>15}")
print("-" * 78)
for p, d, v in params:
    print(f"{p:<28} {d:<32} {v:>15}")

In [ ]:
# 📡 OpenAI 클라이언트로 vLLM 서버 호출
# ⚠️ 실행 전: 터미널에서 vLLM 서버를 먼저 띄우세요 (위 명령어 참고)

import requests, time

VLLM_BASE_URL = "http://localhost:9100/v1"
vllm_server_running = False
print("🔍 vLLM 서버 연결 확인 중...")

for attempt in range(5):
    try:
        resp = requests.get("http://localhost:9100/health", timeout=5)
        if resp.status_code == 200:
            print("✅ vLLM 서버가 실행 중입니다.")
            vllm_server_running = True
            break
    except requests.ConnectionError:
        pass
    if attempt < 4:
        print(f"   재시도 {attempt+1}/5...")
        time.sleep(5)

if not vllm_server_running:
    print("\n❌ vLLM 서버에 연결할 수 없습니다.")
    print("   터미널에서 위 명령어로 서버를 먼저 띄우세요.")
else:
    from openai import OpenAI
    client = OpenAI(base_url=VLLM_BASE_URL, api_key="not-needed")

    models = client.models.list()
    model_id = models.data[0].id
    print(f"📌 서빙 중인 모델: {model_id}")

    start = time.time()
    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": "당신은 유능한 AI 어시스턴트입니다."},
            {"role": "user",   "content": "파이썬의 장점을 3가지 알려주세요."},
        ],
        temperature=0.7,
        max_tokens=256,
    )
    elapsed = time.time() - start

    print(f"\n💬 응답:\n{response.choices[0].message.content}")
    print(f"\n📊 응답 시간: {elapsed:.2f}s")
    print(f"   토큰: 입력 {response.usage.prompt_tokens}, 출력 {response.usage.completion_tokens}")

In [ ]:
# 🌊 스트리밍 응답 — vLLM도 OpenAI와 동일하게 stream=True
if not vllm_server_running:
    print("⚠️ 서버가 실행 중이 아닙니다.")
else:
    print("🌊 스트리밍 응답 테스트")
    print("=" * 50)
    print("❓ 질문: 딥러닝이 무엇인지 초보자에게 설명해주세요.\n")
    print("💬 응답: ", end="", flush=True)

    start = time.time()
    token_count = 0
    first_token_time = None

    stream = client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": "딥러닝이 무엇인지 초보자에게 설명해주세요."}],
        temperature=0.7,
        max_tokens=200,
        stream=True,
    )

    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            if first_token_time is None:
                first_token_time = time.time() - start
            print(delta, end="", flush=True)
            token_count += 1

    total = time.time() - start
    print(f"\n\n📊 성능:")
    print(f"   첫 토큰 (TTFT): {first_token_time:.3f}s")
    print(f"   총 시간:        {total:.2f}s")
    print(f"   토큰 수:        {token_count}개")
    print(f"   처리 속도:      {token_count/total:.1f} tok/s")

### 👥 멀티유저 동시 사용 시뮬레이션 (방식 B가 프로덕션 서빙에 권장되는 이유, 직접 증명)

"vLLM 서버가 멀티유저에 강하다"는 주장을 **실제로 측정해 증명**합니다.

- **가상 사용자 N명**이 각자 **세션**(여러 번의 질문)을 가지고, 서로 다른 시점에 접속(ramp-up)해 **동시에** 서버를 사용합니다.
- 같은 양의 요청을 **(A) 순차(1명씩)** vs **(B) 동시(N명)** 로 처리해 비교합니다.

| 지표 | 의미 |
|------|------|
| 전체 소요시간 | 모든 요청을 끝내는 데 걸린 실측 시간 |
| 처리량 (req/s, tok/s) | 서버가 초당 처리한 요청·토큰 수 |
| 응답시간 p50 / p95 / 최대 | 사용자 체감 지연 분포 (꼬리 지연 포함) |
| 속도 향상 배수 | 동시 처리가 순차 대비 몇 배 빠른지 |

> 💡 클라이언트는 "여러 사용자가 동시에 요청"하는 상황만 만들고, 실제 배칭은 vLLM 서버가
> Continuous Batching으로 자동 수행합니다. 그래서 사용자가 늘어도 처리량이 무너지지 않습니다.

In [ ]:
# 👥 멀티유저 동시 사용 시뮬레이션 — 방식 B(서버)가 멀티유저 서빙에 강함을 직접 증명
# 가상 사용자 N명이 각자 세션(여러 질문)을 갖고, 접속 시점을 달리해(ramp-up) 동시에
# 서버를 사용합니다. 같은 작업을 (B)동시 vs (A)순차(1명씩) 로 처리해 비교합니다.
import time, random, json, os, statistics
import concurrent.futures

if not vllm_server_running:
    print("⚠️ 서버가 실행 중이 아닙니다 — 건너뜀")
else:
    NUM_USERS    = 12     # 동시 접속 사용자 수
    REQS_PER_USER = 3     # 사용자당 질문 수(세션 길이)
    MAX_TOKENS   = 128
    questions = [
        "파이썬의 장점을 3가지 알려줘.", "머신러닝과 딥러닝의 차이는?",
        "REST API가 뭐야?", "도커를 왜 쓰는지 설명해줘.",
        "좋은 커밋 메시지 작성법은?", "TCP와 UDP 차이는?",
        "가비지 컬렉션이 뭐야?", "캐시는 왜 쓰는 거야?",
    ]
    rnd = random.Random(0)

    def ask(prompt):
        t0 = time.time()
        try:
            r = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7, max_tokens=MAX_TOKENS,
            )
            return (time.time() - t0, r.usage.completion_tokens, True)
        except Exception:
            return (time.time() - t0, 0, False)

    def user_session(uid, ramp):
        # 한 사용자가 자기 세션에서 여러 질문을 순차로 던짐
        if ramp:
            time.sleep(rnd.uniform(0, 1.5))   # 접속 시점 분산(ramp-up)
        return [ask(questions[(uid + k) % len(questions)]) for k in range(REQS_PER_USER)]

    def run(concurrent_mode):
        start = time.time()
        if concurrent_mode:                    # N명이 동시에 서버 사용
            with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_USERS) as ex:
                subs = list(ex.map(lambda u: user_session(u, True), range(NUM_USERS)))
            recs = [r for sub in subs for r in sub]
        else:                                  # 한 명씩 순차 처리(baseline)
            recs = [r for u in range(NUM_USERS) for r in user_session(u, False)]
        wall = time.time() - start
        lat = sorted(x[0] for x in recs if x[2])
        tok = sum(x[1] for x in recs if x[2])
        ok  = sum(1 for x in recs if x[2])
        return wall, lat, tok, ok, len(recs)

    total = NUM_USERS * REQS_PER_USER
    print(f"👥 멀티유저 시뮬레이션: 사용자 {NUM_USERS}명 × 각 {REQS_PER_USER}질문 = 총 {total}건")
    print("=" * 72)

    cw, clat, ctok, cok, cn = run(True)    # (B) 동시
    sw, slat, stok, sok, sn = run(False)   # (A) 순차 baseline

    def pct(s, p):
        return s[min(len(s) - 1, int(len(s) * p))] if s else 0.0

    print(f"{'시나리오':<14}{'전체시간':>9}{'req/s':>8}{'tok/s':>8}{'p50':>8}{'p95':>8}{'최대':>8}")
    print("-" * 72)
    print(f"{'동시('+str(NUM_USERS)+'명)':<14}{cw:>8.1f}s{cok/cw:>8.2f}{ctok/cw:>8.0f}"
          f"{pct(clat,.5):>7.2f}s{pct(clat,.95):>7.2f}s{max(clat):>7.2f}s")
    print(f"{'순차(1명씩)':<14}{sw:>8.1f}s{sok/sw:>8.2f}{stok/sw:>8.0f}"
          f"{pct(slat,.5):>7.2f}s{pct(slat,.95):>7.2f}s{max(slat):>7.2f}s")
    print("-" * 72)
    print(f"\n🚀 동시 처리가 순차 대비 {sw/cw:.1f}x 빠름 (총 {total}건 완료 기준)")
    print(f"   ✅ 성공 {cok}/{cn}건 · 같은 GPU 1대로 {NUM_USERS}명 동시 사용 감당")
    print("\n💡 클라이언트는 '동시에 요청'만 했고, 실제 배칭은 vLLM 서버가 Continuous")
    print("   Batching으로 자동 수행 → 사용자가 몰려도 처리량(tok/s)이 유지됩니다.")

    # 💾 결과 저장 + 방식 A 참고
    res = {
        "users": NUM_USERS, "reqs_per_user": REQS_PER_USER, "total_requests": total,
        "concurrent": {"wall_s": round(cw, 1), "req_s": round(cok/cw, 2), "tok_s": round(ctok/cw),
                       "p50_s": round(pct(clat, .5), 2), "p95_s": round(pct(clat, .95), 2),
                       "max_s": round(max(clat), 2), "ok": cok},
        "sequential": {"wall_s": round(sw, 1), "req_s": round(sok/sw, 2), "tok_s": round(stok/sw)},
        "speedup_x": round(sw/cw, 2),
    }
    with open("method_b_result.json", "w") as f:
        json.dump(res, f, ensure_ascii=False, indent=2)
    print("\n💾 결과 저장: method_b_result.json")
    if os.path.exists("method_a_result.json"):
        a = json.load(open("method_a_result.json"))
        print(f"📊 (참고) 방식 A 오프라인 배치 처리량 {a['batch_throughput_tok_s']:.0f} tok/s "
              f"— 단, 방식 A는 단일 프로세스라 외부 멀티유저 동시 호출 자체가 불가합니다.")

### 📊 방식 A vs 방식 B — 실행 결과 숫자 비교 (결론)

`method_a_result.json`(방식 A가 저장한 오프라인 처리량)과, **같은 프롬프트 5개**를 방식 B 서버에 보낸 처리량을 한 표로 직접 비교합니다.

| 항목 | 방식 A (오프라인) | 방식 B (서버) |
|------|------------------|---------------|
| 처리량 (tok/s) | 실측 | 실측 |
| 외부 HTTP 호출 | ❌ | ✅ |
| 동시 여러 사용자 | ❌ | ✅ |
| 모델 상주 | ❌ 매번 로드 | ✅ 한번 로드 |

> 💡 처리량 자체는 둘 다 같은 vLLM 엔진이라 비슷합니다. **방식 B의 가치는 "여러 사용자 동시 서빙"** 이며, 그 정량적 우위는 바로 위 멀티유저 시뮬레이션의 속도 향상 배수로 확인합니다.

In [ ]:
# 📊 방식 A vs 방식 B — 같은 작업을 같은 지표(tok/s)로 직접 숫자 비교
# 방식 A가 저장한 method_a_result.json(오프라인 배치 처리량)과,
# 방식 B 서버에 '동일한 5개 프롬프트'를 보낸 처리량을 나란히 비교합니다.
import json, os, time
import concurrent.futures

if not vllm_server_running:
    print("⚠️ 서버가 실행 중이 아닙니다 — 건너뜀")
elif not os.path.exists("method_a_result.json"):
    print("⚠️ method_a_result.json 이 없습니다.")
    print("   → 커널 재시작 전, 방식 A의 '배칭 성능 비교' 셀을 먼저 실행해 결과를 저장하세요.")
else:
    a = json.load(open("method_a_result.json"))
    same_prompts = [   # 방식 A의 '배칭 비교' 셀에서 쓴 것과 동일한 5개
        "파이썬의 장점을 3가지 알려주세요.",
        "머신러닝과 딥러닝의 차이점은?",
        "한국의 대표적인 전통 음식 3가지는?",
        "클라우드 컴퓨팅이란 무엇인가요?",
        "좋은 코드를 작성하는 방법은?",
    ]

    def ask(p):
        r = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": p}],
            temperature=0.7, max_tokens=128,
        )
        return r.usage.completion_tokens

    # 방식 B: 같은 5개를 서버에 동시 요청 → 처리량 측정
    start = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(same_prompts)) as ex:
        toks = list(ex.map(ask, same_prompts))
    b_t = time.time() - start
    b_thr = sum(toks) / b_t
    a_thr = a["batch_throughput_tok_s"]

    print("📊 방식 A vs 방식 B — 직접 비교 (동일 프롬프트 5개)")
    print("=" * 60)
    print(f"{'항목':<20}{'방식 A(오프라인)':>20}{'방식 B(서버)':>18}")
    print("-" * 60)
    print(f"{'처리량 (tok/s)':<20}{a_thr:>20.1f}{b_thr:>18.1f}")
    print(f"{'외부 HTTP 호출':<20}{'❌ 불가':>19}{'✅ 가능':>17}")
    print(f"{'동시 여러 사용자':<20}{'❌ 단일프로세스':>18}{'✅ 가능':>17}")
    print(f"{'모델 상주(재로딩)':<20}{'❌ 매번 로드':>18}{'✅ 한번 로드':>16}")
    print("-" * 60)
    print("\n💡 해석")
    print("   • 처리량(tok/s)은 A·B 모두 같은 vLLM 엔진이라 비슷합니다.")
    print("   • 진짜 차이는 '여러 사용자를 동시에 서빙할 수 있는가' — 방식 B만 가능합니다.")
    print("   • 멀티유저 부하에서의 동시 처리 우위는 위 '멀티유저 시뮬레이션' 셀의")
    print("     순차 대비 속도 향상 배수로 확인하세요.")

---

## 5️⃣ GPU별 vLLM 추천 설정

GPU VRAM에 따라 서빙 가능한 모델과 권장 설정이 달라집니다.

---

In [ ]:
# 📋 GPU별 vLLM 모델/설정 가이드
print("📋 GPU별 vLLM 추천 설정")
print("=" * 78)

gpu_configs = [
    {
        "gpu": "RTX 4060 (8GB)",
        "models": [("Qwen2.5-1.5B-Instruct", "float16", 1024, 0.80)],
        "note": "1.5B까지 가능, 3B 이상은 VRAM 부족",
    },
    {
        "gpu": "RTX 4090 (24GB)",
        "models": [
            ("Qwen2.5-7B-Instruct",     "float16", 4096, 0.85),
            ("Qwen2.5-7B-Instruct-AWQ", "auto",    8192, 0.85),
        ],
        "note": "7B FP16 또는 7B AWQ 양자화",
    },
    {
        "gpu": "TITAN RTX (24GB)",
        "models": [("Qwen2.5-7B-Instruct", "float16", 2048, 0.85)],
        "note": "Turing 아키텍처 — enforce-eager 필요 (FA2 미지원)",
    },
    {
        "gpu": "A100 (40GB / 80GB)",
        "models": [
            ("Qwen2.5-14B-Instruct",     "float16", 8192, 0.90),
            ("Qwen2.5-72B-Instruct-AWQ", "auto",    4096, 0.90),
        ],
        "note": "프로덕션 권장 GPU",
    },
]

for c in gpu_configs:
    print(f"\n🔹 {c['gpu']}")
    print(f"   {c['note']}")
    for model, dtype, max_len, mem in c["models"]:
        print(f"   → {model} (dtype={dtype}, max_len={max_len}, mem={mem})")

# 현재 GPU에 맞는 추천
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_name = torch.cuda.get_device_name(0)
    print(f"\n" + "=" * 78)
    print(f"📌 현재 GPU: {gpu_name} ({vram_gb:.0f}GB)")
    if vram_gb >= 20:
        print(f"   → 7B FP16 서빙 추천")
    elif vram_gb >= 6:
        print(f"   → 1.5B FP16 서빙 추천")
    else:
        print(f"   → vLLM 대신 Ollama 사용 추천")

---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

| 주제 | 핵심 내용 |
|------|----------|
| PagedAttention | KV cache 메모리 낭비 90%+ 절감 |
| Continuous Batching | 빈 슬롯 즉시 재활용 → GPU 항상 바쁨 |
| 오프라인 추론 | `LLM().generate(prompts, SamplingParams)` |
| API 서버 | `python -m vllm.entrypoints.openai.api_server` |
| OpenAI 호환 | 기존 `openai` 코드 그대로 — `base_url`만 교체 |
| GPU별 설정 | RTX 4060 = 1.5B, RTX 4090/TITAN = 7B, A100 = 14~72B |

### 양자화 + vLLM 페어링 (Session 22~24 연결)

```
Session 22~24:  HF 모델  ──AWQ/GPTQ──▶  4bit 모델
                                            │
Session 25 (지금):                          ▼
                          vLLM (--quantization awq)
                                            │
                                            ▼
                      🚀 production 서빙 (OpenAI 호환 API)
```

### 다음 세션 예고

- 🔜 **Session 26**: RAG 기본개념과 파이프라인 — vLLM으로 띄운 LLM 백엔드 위에서 검색 증강 생성을 구현합니다.

---

### 💡 실습 과제

1. vLLM Python API로 Qwen2.5-1.5B-Instruct 오프라인 추론 실행
2. vLLM API 서버 띄우고 OpenAI 클라이언트로 호출
3. Ollama와 vLLM에 같은 프롬프트 5개씩 보내 응답 시간 비교
4. (심화) `--quantization awq` 옵션으로 AWQ 양자화 모델 서빙 (Session 22~24 결과 활용)

### 📚 참고 자료
- [vLLM 공식 문서](https://docs.vllm.ai/)
- [vLLM GitHub](https://github.com/vllm-project/vllm)
- [PagedAttention 논문](https://arxiv.org/abs/2309.06180)

---

In [ ]:
print("✅ Session 25 완료!")
print("📚 다음 세션: RAG 기본개념과 파이프라인 (Part 4 시작)")